<a href="https://colab.research.google.com/github/LuisGomes25/datasync-Projeto-CRISP-DM-abandono/blob/main/notebooks/04_avaliacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os                                                        # funções do sistema: pastas e caminhos

REPO = "datasync-Projeto-CRISP-DM-abandono"                       # nome da pasta que o clone vai criar
URL = f"https://github.com/LuisGomes25/{REPO}.git"                # endereço público do repositório

if not os.path.exists(REPO) and os.path.basename(os.getcwd()) != REPO:   # só clona se ainda não clonou
    !git clone -q {URL}                                           # copia o repositório para o Colab (-q = sem barulho)

if os.path.basename(os.getcwd()) != REPO:                         # só entra na pasta se ainda não estiver nela
    os.chdir(REPO)                                                # muda a pasta de trabalho para o repositório

!pip install -q shap==0.54.1                                      # instala o SHAP na versão do requirements.txt

os.makedirs("resultados/sprint6", exist_ok=True)                  # cria a pasta de saída (não dá erro se já existir)
print("Pasta atual:", os.getcwd())                                # confirma onde estamos

Pasta atual: /content/datasync-Projeto-CRISP-DM-abandono


In [2]:
import numpy as np                                                # cálculos numéricos
import pandas as pd                                               # tabelas

SEED = 42                                                         # semente: garante o mesmo resultado em toda execução
ETAPAS = ["ensino_medio", "anos_finais", "anos_iniciais"]         # as três etapas, na ordem da Sprint 5

MELHOR_MODELO = {                                                 # vencedores da Sprint 5 (maior PR-AUC)
    "ensino_medio":  "Random_Forest",
    "anos_finais":   "Hist_Gradient_Boosting",
    "anos_iniciais": "Hist_Gradient_Boosting",
}

NUM_BASE = [                                                      # variáveis numéricas do cenário completo da Sprint 5
    "tdi_distorcao", "atu_alunos_turma", "had_horas_aula",
    "dsu_doc_superior", "afd_formacao_adequada", "ied_esforco_alto",
    "icg_complexidade", "ird_regularidade", "inse_media",
    "flag_inse_ausente", "log_matriculas", "pct_noturno",
    "pct_integral", "pct_transporte_publico", "alunos_por_docente",
    "indice_infra", "flag_sem_turma_etapa",
]
CAT_BASE = ["SG_UF", "NO_CATEGORIA", "NO_DEPENDENCIA"]            # variáveis de texto: UF, urbana/rural, rede

df = pd.read_csv(
    "dados/tabela_analitica.csv",                                 # caminho relativo: funciona para qualquer pessoa
    encoding="utf-8-sig",                                         # remove um caractere invisível do início do arquivo
    dtype={"codigo_escola": str, "CO_MUNICIPIO": str},            # códigos como texto (não perdem zeros nem viram número)
    low_memory=False,                                             # lê o arquivo inteiro de uma vez para acertar os tipos
)

assert df.shape == (136713, 27), df.shape                         # trava: para tudo se a base não for a esperada
faltando = [c for c in NUM_BASE + CAT_BASE if c not in df.columns]   # lista colunas que deveriam existir e não existem
assert not faltando, faltando                                     # trava: para tudo se faltar alguma

def criar_alvo(serie, percentil=80):                              # mesma regra da Sprint 5
    limite = float(serie.quantile(percentil / 100))               # valor da taxa no percentil pedido
    if limite == 0.0:                                             # nos Anos Iniciais o P80 é zero...
        return (serie > 0).astype(int), limite                    # ...então "alto" = qualquer abandono
    return (serie >= limite).astype(int), limite                  # demais etapas: taxa no limite ou acima

for etapa in ETAPAS:                                              # confere cada etapa
    sub = df[df["etapa"] == etapa]                                # separa as linhas da etapa
    y, lim = criar_alvo(sub["taxa_abandono"])                     # cria o alvo
    print(f"{etapa:15s} n={len(sub):>6,}  limite={lim:.2f}  alto={y.mean()*100:.2f}%")

ensino_medio    n=21,229  limite=4.40  alto=20.03%
anos_finais     n=44,803  limite=1.80  alto=20.12%
anos_iniciais   n=70,681  limite=0.00  alto=15.28%
